# 《PythAPCS123》單元 13-1：競技程式線上評判系統（Online Judge）運作機制與評判結果型別解析

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_13-1_online_judge_mechanisms_and_verdicts.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
**核心目標**：透視競技程式自動評判系統（Online Judge, OJ）的標準輸入輸出重導向黑盒子架構，徹底搞懂 AC、CE、WA、TLE、RE、MLE 等各項評判代碼的物理意涵與觸發原因，建立專業的除錯心理建設，擺脫「為什麼我電腦跑得過，傳上去卻噴錯」的初學者最大困惑。


### 13.1.1 自動評判系統（Online Judge）運作架構與 stdin/stdout 導向

許多剛接觸 APCS、ZeroJudge 或 LeetCode 的初學者，常常對「線上評判（Online Judge）」感到無比神祕：伺服器到底是如何知道我寫的程式是對是錯的？難道背後有人類裁判坐在螢幕前一行行盯著我的程式碼看嗎？答案當然不是。線上評判系統本質上是一套「全自動化黑盒子黑箱測試（Black-box Testing）程序」，它根本不在乎你的變數叫什麼名字、註解寫得有多漂亮，它唯一檢驗的就是：「給定標準輸入資料，你的程式能否在規定的時間與記憶體額度內，在標準輸出流中產生與官方標準答案（標程答案）一模一樣的位元字串」。

在實體運作上，評判伺服器會將官方準備的大量隱藏測試資料檔（Input Files，例如 `test1.in`, `test2.in`...）透過作業系統的「管道重新導向（Pipe Redirection）」灌入你的程式標準輸入（`sys.stdin`，也就是平常 `input()` 讀取的地方）；接著，伺服器會把你程式透過 `print()` 輸出的所有字元全數攔截並存成暫存檔（`user.out`）；最後，系統調用差異比對工具（如 Linux 的 `diff` 指令），逐字元比對你的輸出與標準輸出檔（`test1.out`）。只要比對結果完全無誤，該筆測資便順利通過。理解這個黑箱重導向機制是所有競技程式除錯的第一步：你必須明確知道，程式不是在跟你聊天，而是與純文字串流打交道。


In [ ]:
# 13.1.1 程式碼演示：模擬 Online Judge 的標準 I/O 重新導向比對機制
import io
import sys

def mock_online_judge(solution_func, test_inputs, expected_outputs):
    '''
    模擬 Online Judge 評判核心：
    1. 攔截標準輸入 sys.stdin，將測試測資字串注入
    2. 攔截標準輸出 sys.stdout，捕捉學生的 print 結果
    3. 嚴格比對 output 與 expected_output
    '''
    print("=== 模擬 Online Judge 評判沙盒啟動 ===")
    all_passed = True
    
    for i, (inp_data, exp_out) in enumerate(zip(test_inputs, expected_outputs), 1):
        # 建立記憶體字串流替換 sys.stdin 與 sys.stdout
        old_stdin = sys.stdin
        old_stdout = sys.stdout
        sys.stdin = io.StringIO(inp_data)
        captured_output = io.StringIO()
        sys.stdout = captured_output
        
        verdict = "AC"
        try:
            # 執行學生提交的解答函式
            solution_func()
            user_res = captured_output.getvalue()
        except Exception as e:
            verdict = f"RE ({type(e).__name__})"
            user_res = ""
        finally:
            # 務必還原標準輸入輸出
            sys.stdin = old_stdin
            sys.stdout = old_stdout
            
        # 比對結果 (若未 RE 則比對文字)
        if verdict == "AC":
            # 去除頭尾換行空白後嚴格比對
            if user_res.strip() == exp_out.strip():
                print(f"  測資 #{i}: [AC] 判定通過！輸出與標準完全相符")
            else:
                verdict = "WA"
                print(f"  測資 #{i}: [WA] 答案錯誤！預期 '{exp_out.strip()}'，實際得到 '{user_res.strip()}'")
                all_passed = False
        else:
            print(f"  測資 #{i}: [{verdict}] 執行崩潰！")
            all_passed = False
            
    print("=== 評判結束：", "全數通過 (ACCEPTED)" if all_passed else "未完全通過", "===\n")

# 示範一個正常的加法解答程式
def correct_solution():
    line = sys.stdin.readline()
    if line:
        a, b = map(int, line.split())
        print(a + b)

sample_inputs = ["3 5\n", "10 20\n", "-5 8\n"]
sample_expected = ["8\n", "30\n", "3\n"]

mock_online_judge(correct_solution, sample_inputs, sample_expected)


### 13.1.1 語法重點回顧與核心觀念提煉

在剛才的程式碼中，我們親手利用 Python 內建的 `io.StringIO` 與 `sys.stdin` / `sys.stdout` 動態替換，還原了 Online Judge 自動評判系統底層的黑箱運作本質。從這段演示中，我們可以提煉出三個極其關鍵的考場思維：

第一，**「沒有人情味的字元比對」**：評判機只是無情地比對文字。因此，任何非題目要求的輸出（例如在 `input()` 內放入提示文字 `"請輸入數字："`，或是順手印出 `"答案是："`），都會被評判機原封不動存入 `user.out`，進而在比對時直接判定為字元不符，導致拿到冤枉的 Wrong Answer（WA）。

第二，**「測資是多筆獨立驗證的」**：一份題目往往包含 5 到 20 筆、甚至上百筆隱藏測試檔案。你的程式必須在每一筆獨立的執行環境中皆產出正確結果，只要有任何一筆測資超時、當機或答案錯誤，該子題（Subtask）就無法拿下滿分。

第三，**「沙盒防護限制」**：在真實的 OJ 上，學生的程式碼是在嚴格受限的虛擬容器（Sandbox）中執行，禁止存取網路、禁止開啟本地非授權檔案、且嚴格限制 CPU 執行時間與可用 RAM 空間。


In [ ]:
# 13.1.1 學生實作練習：實作一個模擬評判狀態評定工具
# 任務說明：請完成 evaluate_testcase(user_output, expected_output)
# 規則：
# 1. 若 user_output 去除尾端空白換行後，與 expected_output 去除尾端空白換行完全相同，回傳 "AC"
# 2. 若兩者不相符，回傳 "WA"
# 3. 若 user_output 中包含 "Error:" 字樣，代表學生程式丟出例外，回傳 "RE"

def evaluate_testcase(user_output: str, expected_output: str) -> str:
    # 請在此處撰寫你的判定邏輯
    pass

# 測試用例
print("測試 1 判定：", evaluate_testcase("100\n", "100"))
print("測試 2 判定：", evaluate_testcase("100 ", "200"))
print("測試 3 判定：", evaluate_testcase("Error: IndexError", "100"))


In [ ]:
# 13.1.1 單元測試驗證
def evaluate_testcase_ans(user_output: str, expected_output: str) -> str:
    if "Error:" in user_output:
        return "RE"
    if user_output.strip() == expected_output.strip():
        return "AC"
    return "WA"

test_cases = [
    ("42\n", "42", "AC"),
    ("hello world", "hello world\n\n", "AC"),
    ("43", "42", "WA"),
    ("Error: ZeroDivisionError occurred", "0", "RE"),
    ("", "empty", "WA")
]

for idx, (u, e, expected_verdict) in enumerate(test_cases, 1):
    actual = evaluate_testcase_ans(u, e)
    assert actual == expected_verdict, f"測資 #{idx} 驗證失敗：預期 {expected_verdict}，得到 {actual}"

print("🎉 13.1.1 所有測試通過！成功掌握 Online Judge 比對本質與評判邏輯！")


### 13.1.2 通過狀態 `AC`（Accepted）：全數隱藏測資完全吻合的判斷標準

在所有競技程式選手的眼裡，最美麗的兩個英文字母莫過於綠色的 `AC`（Accepted）。當系統給出 `AC` 評判時，代表提交的程式碼在該題目設定的全部測試資料集（Test Set）中，每一筆測資都達成三項嚴苛指標：第一，在指定時限（Time Limit，通常為 1.0 秒）內自然結束；第二，記憶體消耗未超過安全上限（Memory Limit，通常為 64MB ~ 512MB）；第三，標準輸出內容與官方標準答案經比對後「毫無二致」。

然而，許多學生在解題時常常產生一種危險的心理錯覺：「我在本機終端機把題目敘述中的『範例輸入（Sample Input）』打進去，印出來的結果跟『範例輸出（Sample Output）』一模一樣，為什麼送上去沒有 AC？」我們必須牢記：範例測資通常只是出題老師為了方便考生理解題意而提供的「最基本、最陽春」的說明案例。在真正的後台測資庫中，出題團隊會精心埋伏各式各樣的極限與極端邊界資料（Corner Cases），例如：數值大小達到 $10^{18}$ 的大整數、長度達到 $10^5$ 的極長字串、只有單一元素的串列、全數為負數或包含零的極端值、甚至故意構造能觸發最差時間複雜度的殺手測資（Anti-hash 或 Anti-quicksort 數列）。因此，能夠通過範例測資僅代表你看懂了題目的表面規則，唯有思考周延、時間複雜度達標、邊界防禦滴水不漏的程式碼，才能真正征服所有隱藏測資，斬獲綠色滿分 AC！


In [ ]:
# 13.1.2 程式碼演示：範例測資能過，但在極端隱藏測資下翻車的典型範例
def naive_solve(nums):
    '''
    題目要求：給定非空整數串列，找出最大值。
    新手常見盲點：以為數字都是正整數，誤設初值 max_val = 0
    '''
    max_val = 0
    for x in nums:
        if x > max_val:
            max_val = x
    return max_val

def robust_solve(nums):
    '''
    穩健解法：以第一個元素作為初值，或使用負無窮大 float('-inf')
    能安全應對全為負數、包含零等所有極端邊界測資
    '''
    max_val = nums[0]
    for x in nums[1:]:
        if x > max_val:
            max_val = x
    return max_val

# 範例測資 (Sample Input): 數字全為正數
sample_test = [3, 7, 2, 9, 5]
print("--- 1. 範例測資測試 ---")
print("naive_solve 輸出 :", naive_solve(sample_test), "(預期 9, 結果正確！)")
print("robust_solve 輸出:", robust_solve(sample_test), "(預期 9, 結果正確！)")

# 官方隱藏測資 (Hidden Secret Test): 數字全為負數！
hidden_test = [-15, -8, -23, -4, -42]
print("\n--- 2. 官方隱藏測資測試 (全負數極端情況) ---")
naive_res = naive_solve(hidden_test)
robust_res = robust_solve(hidden_test)
print("naive_solve 輸出 :", naive_res, f"-> {'[AC]' if naive_res == -4 else '[WA] 慘遭滑鐵盧！'}")
print("robust_solve 輸出:", robust_res, f"-> {'[AC] 完美過關！' if robust_res == -4 else '[WA]'}")


### 13.1.2 語法重點回顧與核心觀念提煉

剛才的案例生動地揭露了競技程式的殘酷現實：初學者寫出的 `naive_solve` 在範例測資上表現得天衣無縫，但出題者只要準備一組「全為負數」的隱藏測資，原本自信滿滿的程式就會因為初始值 `max_val = 0` 的預設漏洞而立刻噴出錯誤答案（得到 0 而非 -4），直接痛失所有隱藏分數。

要邁向穩定斬獲 AC 的境界，必須建立「隱藏測資沙盤推演」的習慣：
1. **極值邊界檢查**：如果數值都是負數怎麼辦？如果有 0 怎麼辦？數值達到整數上限怎麼辦？
2. **長度邊界檢查**：輸入序列長度 $N=1$ 時能否正常輸出？如果輸入為空能否安全防禦？
3. **初值設定原則**：尋求極大值時，初值應為序列第一個元素 `nums[0]` 或負無窮大 `float('-inf')`；尋求極小值時，應設為 `nums[0]` 或正無窮大 `float('inf')`，絕不可主觀假設數值必定大於零。


In [ ]:
# 13.1.2 學生實作練習：修復只過範例測資的脆弱函式
# 題目說明：給定一個整數串列，回傳其中的「最小正整數」（大於 0 的最小值）。
# 若串列中不存在任何正整數，應回傳 -1。
# 脆弱的範例版本在面對負數與無解情況時會出錯，請完成強韌版 find_min_positive(nums)。

def find_min_positive(nums: list[int]) -> int:
    # 請在此處撰寫可通過全部極端測資的程式碼
    pass

# 測試用例
print("測試 1 (一般混和)：", find_min_positive([4, -2, 7, 1, 9])) # 預期 1
print("測試 2 (全為負數)：", find_min_positive([-5, -10, -3]))     # 預期 -1
print("測試 3 (含零與重複)：", find_min_positive([0, 5, 2, 2]))     # 預期 2


In [ ]:
# 13.1.2 單元測試驗證
def find_min_positive_ans(nums: list[int]) -> int:
    positives = [x for x in nums if x > 0]
    return min(positives) if positives else -1

test_suites = [
    ([4, -2, 7, 1, 9], 1),
    ([-5, -10, -3], -1),
    ([0, 5, 2, 2], 2),
    ([100], 100),
    ([-1], -1),
    ([0, 0, 0], -1)
]

for idx, (data, expected) in enumerate(test_suites, 1):
    res = find_min_positive_ans(data)
    assert res == expected, f"測資 #{idx} 失敗：輸入 {data}，預期 {expected}，得到 {res}"

print("🎉 13.1.2 所有測試通過！成功建立防禦極端隱藏測資的 AC 思維！")


### 13.1.3 語法錯誤 `CE`（Compile Error / SyntaxError）：直譯器拒絕啟動硬傷

在提交程式碼至評判系統後，若系統跳出鮮黃色或紅色的 `CE`（Compile Error），代表你的程式碼在「語法結構上根本無法通過直譯器或編譯器的第一道門禁檢查」。雖然 Python 屬於直譯式語言（Interpreted Language），不需要像 C++ 那樣經過嚴謹的獨立編譯產出執行檔，但 Python 在執行你的任何一行指令之前，直譯器依舊會進行第一階段的「語法剖析與位元組編譯（Bytecode Compilation Phase）」。只要在整份程式碼中存在任何一個括號沒關好、縮排錯位、或是漏打冒號，Python 直譯器在啟動前就會立刻拋出 `SyntaxError`、`IndentationError` 或 `TabError`，連第 1 行程式碼都還沒開始跑就宣告當場死亡！

在考場上拿到 CE 是最令人扼腕的事情，因為這代表程式連參與評判的資格都沒有，直接領 0 分。常見造成 CE 的原因包括：
1. **結構性符號遺漏**：`if`, `for`, `while`, `def` 結尾漏掉英文冒號 `:`。
2. **括號/引號不成對**：左括號 `(` 或引號 `'`, `"` 未閉合，導致直譯器在下一行甚至檔案結尾處困惑報錯。
3. **縮排層次混亂**：同一區塊內縮排空格數忽大忽小，或不小心混用了 Tab 與空白鍵。
4. **關鍵字拼錯或非法字元**：例如將 `while` 拼成 `whlie`，或是輸入法未切換回半形，不慎鍵入全形逗號 `，` 或全形冒號 `：`。在接下來的演示中，我們將透過 Python 動態編譯器深入體驗直譯器如何抓出語法硬傷。


In [ ]:
# 13.1.3 程式碼演示：模擬直譯器語法檢查期捕捉 CE (SyntaxError)
def check_syntax(code_string):
    '''
    利用 Python 內建 compile() 函式模擬直譯器啟動前的語法編譯檢查。
    若語法不合規，將捕捉 SyntaxError 並提取出精確的錯誤行號與字元位置。
    '''
    try:
        # mode='exec' 代表將字串視為完整 Python 模組進行編譯
        compile(code_string, filename="<user_submission>", mode="exec")
        print("  [編譯成功] 語法結構完全合法，直譯器允許啟動！")
        return "PASS"
    except SyntaxError as e:
        print(f"  [CE 捕捉] 檔案: {e.filename}, 行號: {e.lineno}, 偏移字元: {e.offset}")
        print(f"  -> 錯誤描述: {e.msg}")
        print(f"  -> 問題程式碼: {e.text.strip() if e.text else '無法讀取'}")
        pointer = " " * (e.offset - 1) + "^" if e.offset else "^"
        print(f"  -> 定位游標  : {pointer}\n")
        return "CE"

# 測試 1: 合法程式碼
code_valid = "x = 10\nif x > 5:\n    print(x)"
print("測試 1 (正常代碼)：")
check_syntax(code_valid)

# 測試 2: 經典失誤——if 結尾漏掉冒號
code_missing_colon = "x = 10\nif x > 5\n    print(x)"
print("測試 2 (漏掉冒號)：")
check_syntax(code_missing_colon)

# 測試 3: 致命失誤——不慎鍵入中文全形括號或冒號
code_fullwidth = "print('Hello world'）"  # 結尾為全形中文右括號 ）
print("測試 3 (全形符號混入)：")
check_syntax(code_fullwidth)


### 13.1.3 語法重點回顧與核心觀念提煉

透過剛才 `compile()` 的模擬，我們可以清楚看見 Python 直譯器在執行前對語法結構的嚴格審查機制。當直譯器回報 `SyntaxError` 時，它會貼心地指出兩項核心情報：
1. **錯誤行號（`lineno`）**：指出直譯器是在哪一行「撞牆停下」的。值得注意的是，如果上一行的括號沒有閉合（例如 `a = (1 + 2` 漏掉右括號），直譯器通常會報錯在「下一行」甚至更後面的程式碼，因為它一直在等待括號關閉！
2. **游標箭頭（`offset` 與 `^`）**：精準指向它解析到哪個字元時發現結構不對勁。

在考場中，避免 CE 的最佳心法就是「落實半形輸入環境」與「語法配對習慣」：每當鍵入括號 `()`、方括號 `[]` 或引號 `""` 時，務必成對輸入；在每句控制流（`if/elif/else/for/while/def`）結尾，手指都要反射性確認是否有冒號 `:`。只要能通過編譯期檢查，就能確保絕不拿到 0 分 CE。


In [ ]:
# 13.1.3 學生實作練習：語法除錯診斷器
# 任務說明：請修復以下包含三處語法硬傷的程式碼字串，使其能夠順利通過 compile 語法編譯
# 提示：尋找漏冒號、全形字元或未閉合括號

buggy_code = '''
def calculate_average(nums):
    total = sum(nums)
    if len(nums) > 0
        return total / len(nums)
    else:
        return 0
'''

# 請將修正後的合法程式碼存入 fixed_code 字串中
fixed_code = '''
# 請修正後貼在這邊
'''

# 驗證
print("修復後語法檢驗結果：")
check_syntax(fixed_code)


In [ ]:
# 13.1.3 單元測試驗證
correct_code_sample = '''
def calculate_average(nums):
    total = sum(nums)
    if len(nums) > 0:
        return total / len(nums)
    else:
        return 0
'''

try:
    compile(correct_code_sample, filename="<test>", mode="exec")
    comp_ok = True
except SyntaxError:
    comp_ok = False

assert comp_ok, "範例修復代碼應能順利編譯！"
print("🎉 13.1.3 所有測試通過！成功建立預防與排查 CE 語法錯誤的直覺！")


### 13.1.4 答案錯誤 `WA`（Wrong Answer）：輸出結果或格式與官方標程不符分析

`WA`（Wrong Answer，答案錯誤）是競技程式中最常見、也最讓人抓狂的評判結果。拿到 WA 代表你的程式結構完全合法（沒有 CE）、在中途也沒有當機崩潰（沒有 RE）、更在規定時限內完成了計算（沒有 TLE），然而，你的輸出內容在與官方標程比對時，「哪怕只有一個字元不同」，系統都會無情地給出紅色的 WA！

造成 WA 的核心主因可歸類為兩大流派：
第一流派是**「核心計算邏輯錯誤（Semantic Logic Bug）」**：
- 演算法推導不完整：漏掉了題目的特例規則（例如：題目說明兩數相同時應輸出較小編號，程式卻反過來）。
- 邊界偏差（Off-by-one）：迴圈少跑了一輪、大於 `>` 誤寫為大於等於 `>=`、或 `range(1, n)` 漏算了第 `n` 項。
- 運算優先級與除法誤用：將整除 `//` 誤打成小數除法 `/`，或是 `a + b // 2` 漏括號變成 `a + (b // 2)`。

第二流派是初學者最常含冤莫白流淚的**「輸出格式不對齊（Presentation / Format Misalignment）」**：
- 行末多餘空白或漏換行：題目要求各數字以空格分開，行末不留空格，程式卻在最後一個數字後多印了 `' '`。
- 大小寫不合：題目要求輸出大寫 `"YES"`，程式卻印出 `"Yes"` 或 `"yes"`。
- 浮點數格式差之毫釐：題目要求四捨五入輸出到小數點後兩位 `f"{ans:.2f}"`，程式卻直接輸出原生浮點數 `3.1415926`。只要有一絲差異，在自動評判系統眼中都是百分之百的 WA！


In [ ]:
# 13.1.4 程式碼演示：格式細節引發的隱形 WA 陷阱
def compare_output_diff(user_str, official_str):
    '''
    精準展示字元級比對差異：將隱形字元（如空格與換行）可視化
    '''
    print(f"學生輸出: {repr(user_str)}")
    print(f"官方標程: {repr(official_str)}")
    
    if user_str == official_str:
        print("-> [AC] 完全吻合！\n")
    else:
        print("-> [WA] 比對失敗！字元差異分析：")
        min_len = min(len(user_str), len(official_str))
        diff_found = False
        for idx in range(min_len):
            if user_str[idx] != official_str[idx]:
                print(f"   第 {idx} 個字元不同: 學生得到 {repr(user_str[idx])}，官方預期 {repr(official_str[idx])}")
                diff_found = True
                break
        if not diff_found and len(user_str) != len(official_str):
            print(f"   字串長度不同！學生長度 {len(user_str)}，官方長度 {len(official_str)} (可能有多餘行末空格或換行)")
        print()

# 案例一：大小寫差異 (Case Sensitivity)
compare_output_diff("yes\n", "YES\n")

# 案例二：整數 vs 浮點數輸出 (4 vs 4.0)
compare_output_diff("4.0\n", "4\n")

# 案例三：行末多餘空格 (Trailing Space)
compare_output_diff("1 2 3 \n", "1 2 3\n")


### 13.1.4 語法重點回顧與核心觀念提煉

在剛才的字元級比對工具中，我們使用了 Python 的 `repr()` 函式將字串中的隱形字元（例如換行符號 `\n`、空格 `' '`）以最真實的面貌暴露出來。從三個案例中，我們可以學到預防 WA 的三大鋼鐵法則：

1. **大小寫嚴格一致**：在競賽程式中，字串的比對是區分大小寫的（Case-sensitive）。若題目要求印出 `"None"`、`"IMPOSSIBLE"` 或 `"INVALID"`，務必從題目敘述中直接複製字串，切勿自行憑記憶敲鍵盤，避免因拼錯或大小寫不合導致全體 WA。
2. **數值型態輸出精確控制**：若題目計算要求輸出整數，務必透過 `int()` 確保無小數點殘留（如整除應使用 `//` 而非 `/`，避免輸出 `4.0` 造成 WA）。
3. **善用 `join()` 與 `*` 解包消除行末空格**：若需將串列輸出為空格分隔，使用 `print(*ans)` 或是 `' '.join(map(str, ans))`，Python 會自動在元素之間插入單一空格，且行末絕不會殘留多餘空格，是最安全乾淨的標準輸出姿勢。


In [ ]:
# 13.1.4 學生實作練習：安全輸出格式修復器
# 題目說明：給定一個成績清單 grades = [85, 92, 78, 90]
# 題目輸出規範要求：
# 1. 輸出總平均，四捨五入保留到整數（輸出整數格式，不可帶小數點）
# 2. 第二行印出所有及格（>= 60）的成績，由大到小排序，彼此以單一空格分開，行末不留空格
# 請實作 format_report(grades) 並回傳符合規範的字串

def format_report(grades: list[int]) -> str:
    # 請在此處撰寫符合嚴格格式的輸出邏輯
    pass

# 測試用例
print("=== 格式測試輸出 ===")
res = format_report([85, 92, 78, 90])
print(repr(res))


In [ ]:
# 13.1.4 單元測試驗證
def format_report_ans(grades: list[int]) -> str:
    avg = round(sum(grades) / len(grades))
    passing = sorted([g for g in grades if g >= 60], reverse=True)
    pass_str = " ".join(map(str, passing))
    return f"{avg}\n{pass_str}"

test_grades = [85, 92, 78, 90]
expected_str = "86\n92 90 85 78"
assert format_report_ans(test_grades) == expected_str, "成績報告格式不符預期！"

print("🎉 13.1.4 所有測試通過！精通排除 WA 的字元級嚴謹輸出規範！")


### 13.1.5 超時與記憶體超限：`TLE`（Time Limit Exceeded）與 `MLE`（Memory Limit Exceeded）

在線上評判的世界中，評斷一段程式的好壞不只看它算得「對不對」，更看它算得「快不快、省不省」。這就是資源限制評判碼的由來：
- `TLE`（Time Limit Exceeded，超時）：代表你的程式在評判伺服器上連續運轉超過了題目設定的時間極限（通常為 1.0 秒，部分巨量測資題為 2.0 秒或 3.0 秒），被作業系統強制擊殺終止。
- `MLE`（Memory Limit Exceeded，記憶體超限）：代表你的程式在執行過程中申請了過多記憶體空間（例如開了巨大的二維陣列或無節制快取），突破了題目的 RAM 額度上限（通常為 256MB 或 512MB），遭到系統記憶體守護程序中斷。

在 APCS 考場上，**TLE 的出現頻率遠遠高於 MLE**。99% 的初學者遭遇 TLE，主要都是因為演算法的時間複雜度與題目給定的測資規模（$N$ 的大小）完全不匹配所致！在現代電腦硬體下，Python 直譯器在 1 秒之內大約只能執行 $10^7$ 次（約一千萬次）基本運算。如果題目給定的測資規模 $N = 10^5$：
- 若你寫出 $O(N)$ 的單層迴圈，總運算次數為 $10^5$ 次，耗時約 0.02 秒，輕鬆斬獲 AC！
- 若你直覺地寫出雙層巢狀迴圈（$O(N^2)$），總運算次數高達 $(10^5)^2 = 10^{10}$ 次（一百億次），在 Python 下大約需要狂跑整整 15 分鐘！伺服器在跑滿 1.0 秒的瞬間就會毫不猶豫地拔掉插頭，噴出刺眼的 TLE！


In [ ]:
# 13.1.5 程式碼演示：$O(N^2)$ 暴力演算法 vs $O(N)$ 高效演算法的執行時間天壤之別
import time

def slow_two_sum_check(arr, target):
    '''
    $O(N^2)$ 雙層暴力迴圈：檢查陣列中是否存在兩數之和等於 target
    '''
    n = len(arr)
    for i in range(n):
        for j in range(i + 1, n):
            if arr[i] + arr[j] == target:
                return True
    return False

def fast_two_sum_check(arr, target):
    '''
    $O(N)$ 雜湊集合演算法：利用 set 查找只需 O(1) 的特性
    '''
    seen = set()
    for x in arr:
        if target - x in seen:
            return True
        seen.add(x)
    return False

# 建立規模為 N = 5,000 的資料
N = 5000
test_data = list(range(1, N + 1))
target_val = -1  # 故意尋找不存在的目標，觸發最差情況走訪完整資料

print(f"=== 效能實測：資料規模 N = {N} ===")

# 測試 O(N^2) 演算法耗時
t0 = time.time()
slow_two_sum_check(test_data, target_val)
t_slow = time.time() - t0
print(f"1. 雙層暴力 O(N^2) 耗時: {t_slow:.4f} 秒 (當 N=100,000 時將耗時數十分鐘造成 TLE！)")

# 測試 O(N) 演算法耗時
t0 = time.time()
fast_two_sum_check(test_data, target_val)
t_fast = time.time() - t0
print(f"2. 雜湊集合 O(N)   耗時: {t_fast:.6f} 秒 (極速秒殺，穩拿 AC！)")


### 13.1.5 語法重點回顧與核心觀念提煉

看著剛才兩套演算法的耗時對比，相信你已經深刻感受到時間複雜度在競技程式中的生殺大權：僅僅在 $N=5,000$ 的中小型資料規模下，$O(N^2)$ 暴力法就已經明顯感受到遲鈍，若是在 APCS 真題動輒 $N=10^5$ 的極限測資下，暴力迴圈必死無疑。

在考場中預防與除錯 TLE，請隨時牢記「**一秒一千萬次法則（1 second $\approx 10^7$ operations）**」：
1. **先看測資範圍 $N$**：
   - 若 $N \le 20$：可容許指數級 $O(2^N)$ 窮舉。
   - 若 $N \le 500$：$O(N^3)$ 演算法可驚險過關。
   - 若 $N \le 3,000$：$O(N^2)$ 雙層迴圈可穩定過關。
   - 若 $N \le 2 \times 10^5$：**必須使用 $O(N \log N)$ 排序或 $O(N)$ 線性演算法**！
   - 若 $N \ge 10^7$：必須使用 $O(\log N)$ 二分搜尋或 $O(1)$ 數學解。
2. **警惕致命無窮迴圈**：除了複雜度太高外，`while` 迴圈漏寫計數器累加（例如忘了寫 `i += 1`），導致條件永遠為真，也是引發 TLE 的經典低級失誤！


In [ ]:
# 13.1.5 學生實作練習：判斷時間複雜度是否會 TLE
# 任務說明：實作 predict_tle(n, complexity_type, time_limit_sec=1.0)
# 估算規則（以 1 秒上限 10^7 次運算為基準）：
# 1. 若 complexity_type == "O(N)"，預估運算次數為 n
# 2. 若 complexity_type == "O(N^2)"，預估運算次數為 n * n
# 3. 若預估運算次數 > time_limit_sec * 10^7，回傳 "TLE"，否則回傳 "PASS"

def predict_tle(n: int, complexity_type: str, time_limit_sec: float = 1.0) -> str:
    # 請在此處撰寫預估邏輯
    pass

# 測試用例
print("N=10^5 搭配 O(N^2)   :", predict_tle(100000, "O(N^2)"))  # 預期 TLE
print("N=10^5 搭配 O(N)     :", predict_tle(100000, "O(N)"))    # 預期 PASS
print("N=2000 搭配 O(N^2)   :", predict_tle(2000, "O(N^2)"))    # 預期 PASS


In [ ]:
# 13.1.5 單元測試驗證
def predict_tle_ans(n: int, complexity_type: str, time_limit_sec: float = 1.0) -> str:
    limit_ops = time_limit_sec * 10**7
    if complexity_type == "O(N)":
        ops = n
    elif complexity_type == "O(N^2)":
        ops = n * n
    else:
        ops = n
    return "TLE" if ops > limit_ops else "PASS"

assert predict_tle_ans(100000, "O(N^2)") == "TLE"
assert predict_tle_ans(100000, "O(N)") == "PASS"
assert predict_tle_ans(2000, "O(N^2)") == "PASS"
assert predict_tle_ans(5000, "O(N^2)") == "TLE"

print("🎉 13.1.5 所有測試通過！成功建立演算法量級與 TLE 預防直覺！")


### 13.1.6 執行崩潰 `RE`（Runtime Error）：程式中途暴斃成因與「環境差異」迷思破除

當線上評判回傳深紅色的 `RE`（Runtime Error，執行時期錯誤）時，代表你的程式碼已經通過了語法編譯，但在處理某一筆隱藏測試資料時，執行到一半突然引發了未被處理的嚴重例外（Exception），導致作業系統直接強行拋出錯誤信號並中斷程式！

初學者面對 RE，最常脫口而出的一句話就是：「怎麼可能？我在自己的 VS Code 或 Colab 上跑得好好的，輸入數字答案都對啊，是不是評判伺服器壞掉了？」這就是經典的**「環境差異迷思」**。在你的本機電腦上跑得過，僅僅是因為你輸入的測資太過溫和、太過理想！評判系統的隱藏測資具備高度的惡意與極限性，只要你的程式存在一絲邏輯漏洞，就會立刻引爆。

在競技程式中，最常見的四大 RE 元兇排行榜如下：
1. **`IndexError: list index out of range`**（永遠的殺手榜首！）：陣列取索引越界，例如長度為 3 的串列試圖存取 `arr[3]`，或是對空串列存取 `arr[0]`。
2. **`ValueError`**：資料型態轉換失敗，例如將文字或空行硬轉為整數 `int("")`，或是解包 `a, b = map(...)` 時輸入項數不符合預期。
3. **`ZeroDivisionError`**：分母為零，在除法 `a / b` 或模除 `a % b` 時分母為 0。
4. **`RecursionError: maximum recursion depth exceeded`**：遞迴終止條件（Base case）寫錯，導致無限呼叫爆堆疊（超過 Python 預設 1000 層限制）。在下面的實作中，我們將模擬常見的 RE 崩潰並學習如何從容排查。


In [ ]:
# 13.1.6 程式碼演示：重現考場常見 RE (Runtime Error) 崩潰現場
def simulate_re_sandbox(user_function, test_cases):
    '''
    模擬競技平台執行沙盒，捕捉各種執行崩潰原因
    '''
    print("=== 開始執行沙盒測試 ===")
    for i, test in enumerate(test_cases, 1):
        try:
            res = user_function(test)
            print(f"  測資 #{i}: 輸入 {test} -> 成功執行，輸出 {res}")
        except IndexError as e:
            print(f"  測資 #{i}: 輸入 {test} -> [RE 爆發] IndexError! 陣列索引越界: {e}")
        except ZeroDivisionError as e:
            print(f"  測資 #{i}: 輸入 {test} -> [RE 爆發] ZeroDivisionError! 數學除以零: {e}")
        except ValueError as e:
            print(f"  測資 #{i}: 輸入 {test} -> [RE 爆發] ValueError! 數值轉型失敗: {e}")
        except Exception as e:
            print(f"  測資 #{i}: 輸入 {test} -> [RE 爆發] 未知例外: {type(e).__name__} ({e})")
    print("=== 沙盒測試結束 ===\n")

# 示範一個脆弱的程式碼：計算清單的第一個元素除以第二個元素
def fragile_division(lst):
    return lst[0] // lst[1]

# 測試用例：包含正常案例與三種典型會引發 RE 的惡意邊界測資
nasty_test_cases = [
    [10, 2],    # 正常: 輸出 5
    [10],       # 致命 1: 長度不足，lst[1] 觸發 IndexError!
    [10, 0],    # 致命 2: 分母為 0，lst[0] // lst[1] 觸發 ZeroDivisionError!
    []          # 致命 3: 空串列，lst[0] 觸發 IndexError!
]

simulate_re_sandbox(fragile_division, nasty_test_cases)


### 13.1.6 語法重點回顧與核心觀念提煉

看著剛才沙盒中的紅字警報，你應該已經恍然大悟：並不是評判系統壞了，而是我們的程式碼缺乏對「邊界特例」的免疫力！在真實的考場中，測資絕對不會乖乖照著理想情況輸入，任何可能的空串列、單一元素、零分母都有可能藏匿在隱藏測資中。

化解 RE 的最高指導原則是**「防禦性編程（Defensive Programming）」**：
1. **存取索引前先防守**：在讀取 `arr[i]` 之前，永遠確保 `0 <= i < len(arr)`；在取出 `lst[0]` 前，先檢查 `if len(lst) > 0`。
2. **除法運算前檢驗分母**：在執行 `/` 或 `%` 前，先以 `if denom != 0:` 築起第一道防線。
3. **字串切割前檢驗長度**：在對一行字串進行解包 `a, b = line.split()` 時，若該行為空或項目不足，解包就會直接噴出 ValueError。在後續單元中，我們將學會更進階的例外捕捉技巧，打造銅牆鐵壁般的穩健程式！


In [ ]:
# 13.1.6 學生實作練習：打造零崩潰的防禦型安全除法器
# 任務說明：請實作 safe_division(lst)
# 規則：
# 1. 若 lst 長度小於 2，回傳 "INVALID_LENGTH"
# 2. 若分母 lst[1] 為 0，回傳 "DIVISION_BY_ZERO"
# 3. 若皆正常，回傳 lst[0] // lst[1] 的整數商
# 確保在面對任何惡意輸入時絕不噴出 RE 崩潰！

def safe_division(lst: list[int]):
    # 請在此處撰寫安全防禦邏輯
    pass

# 測試用例
print("測試 1 (正常):", safe_division([20, 4]))   # 預期 5
print("測試 2 (長度不足):", safe_division([10]))  # 預期 INVALID_LENGTH
print("測試 3 (除以零):", safe_division([10, 0]))  # 預期 DIVISION_BY_ZERO
print("測試 4 (空串列):", safe_division([]))       # 預期 INVALID_LENGTH


In [ ]:
# 13.1.6 單元測試驗證
def safe_division_ans(lst: list[int]):
    if len(lst) < 2:
        return "INVALID_LENGTH"
    if lst[1] == 0:
        return "DIVISION_BY_ZERO"
    return lst[0] // lst[1]

test_data = [
    ([20, 4], 5),
    ([10], "INVALID_LENGTH"),
    ([10, 0], "DIVISION_BY_ZERO"),
    ([], "INVALID_LENGTH"),
    ([0, 5], 0),
    ([-8, 2], -4)
]

for idx, (inp, exp) in enumerate(test_data, 1):
    res = safe_division_ans(inp)
    assert res == exp, f"測資 #{idx} 失敗：輸入 {inp}，預期 {exp}，得到 {res}"

print("🎉 13.1.6 所有測試通過！成功掌握防範 RE 崩潰的防禦性編程要訣！")


## 單元總結與自我評量

### 專業除錯地圖：6 大評判狀態代碼總整理

在競技程式設計與 APCS 考場上，看懂系統回傳的狀態代碼是快速定位臭蟲的第一步。請隨時參照下表進行除錯診斷：

| 評判代碼 | 英文全稱 | 中文含意 | 核心根本原因 | 第一時間急救處方 |
| :---: | :--- | :--- | :--- | :--- |
| **`AC`** | Accepted | 通過 | 答案正確、時空達標、格式無誤 | 恭喜拿下滿分！進入下一題！ |
| **`CE`** | Compile Error | 語法/編譯錯誤 | 漏冒號、括號不對稱、混用全形符號或縮排錯位 | 檢視報錯行號與指標 `^`，排查符號與縮排 |
| **`WA`** | Wrong Answer | 答案錯誤 | 演算法邏輯盲點、邊界偏差、多印空格提示字 | 自編極端測資推演、比對大小寫與空格換行 |
| **`TLE`** | Time Limit Exceeded | 時間超限（逾時） | 複雜度太高（如 $O(N^2)$）、陷入無窮迴圈 | 估算 $N$ 量級、改用 $O(N)$ 雜湊或排序、檢查 `while` 累加 |
| **`MLE`** | Memory Limit Exceeded | 記憶體超限 | 宣告過大陣列、無限堆疊快取耗盡 RAM | 避免複製巨大串列、改用串流處理（Generator） |
| **`RE`** | Runtime Error | 執行時期崩潰 | 索引越界（IndexError）、除以零、轉型失敗 | 檢查空串列、邊界索引防守（`if 0 <= i < len`） |

---

### 自我實力檢測清單
- [ ] 我能理解 Online Judge 底層「標準輸入輸出字元重導向」的黑盒子比對本質。
- [ ] 我明白為什麼不能在 `input()` 中放入提示文字，以及為什麼輸出多一個空格會導致 WA。
- [ ] 我知道範例測資通過不代表萬事大吉，能主動設想全負數、包含零、單一元素等隱藏極端測資。
- [ ] 我能看懂直譯器拋出 `SyntaxError` 時的行號與 `^` 指標，並知道上一行未閉合括號的陷阱。
- [ ] 我熟記「1 秒一千萬次運算（$10^7$ ops）」準則，能根據 $N$ 的大小預先評估是否會發生 TLE。
- [ ] 我能辨識造成 RE 崩潰的四大天王（IndexError、ValueError、KeyError、ZeroDivisionError）並落實防禦性編程。
